# Silver → Gold · V-Credit

## Objetivo
Modelar a camada Gold em formato Star Schema para consumo no Power BI, construir tabelas de indicadores agregados (KPIs) e consolidar o log de qualidade do pipeline.

## Estrutura de entregáveis

- **Dimensões:** `dim_calendario`, `dim_cliente`, `dim_motivo`, `dim_atendente` e `dim_canal`
- **Fatos:** `fact_chamados` (nível chamado) e `fact_jornadas` (nível jornada)
- **KPIs:** `kpi_mensal`, `kpi_funil_motivo` e `kpi_atendente_mensal`
- **Qualidade:** `log_qualidade` (consolidação Silver + Gold)
- **Exports:** Geração de arquivos CSV padronizados no Volume `exports`

## Regras e premissas de modelagem
- **Jornadas:** agrupamento por cliente e motivo; inícios definidos por autosserviço, primeiro chamado do motivo ou janela inativa superior a 24 horas (`JANELA_MAX_HORAS`).
- **Canal e Fluxo:** a dimensão `dim_canal` possui granularidade canal × fluxo (Telefone/Digital), herdada do canal de entrada da jornada.
- **Membros de Exceção:** inclusão dos SKs `-1` (Autosserviço) e `-2` (Atendente desconhecido/inválido) na `dim_atendente`.
- **Preservação de NULLs:** valores ausentes de custo, nota e tempos não são substituídos por zero.
- **Privacidade (LGPD):** remoção de dados pessoais identificáveis (`nome` e `email`) da camada Gold.
- **Indicadores Reagregáveis:** manutenção dos numeradores e denominadores em tabelas agregadas para viabilizar filtros no Power BI.

## 1. Configurações, parâmetros e funções auxiliares

- **Parâmetros de negócio:** definição explícita de janelas temporais, limites de complexidade e volumes mínimos
- **gravar_gold:** salva dataframes em formato Delta na camada Gold com modo overwrite e esquema dinâmico
- **com_chave:** gera chave primária substituta (Surrogate Key - `sk_*`) determinística via função de janela
- **razao:** cálculo seguro de proporções evitando divisão por zero (retorna NULL caso o denominador seja 0)
- **salvar_log_gold e exigir_gold:** rotinas de registro e validação bloqueante da qualidade de dados na camada Gold

In [0]:
from datetime import date
from pyspark.sql import functions as F, Window

catalogo = "vcredit"
silver = f"{catalogo}.silver"
gold = f"{catalogo}.gold"
caminho_exports = f"/Volumes/{catalogo}/gold/exports/"

# Parâmetros de negócio (premissas explícitas)
JANELA_MAX_HORAS = 24            # salvaguarda: chamado humano após mais de 24h inicia nova jornada
LIMITE_COMPLEXA = 0.45           # N2 / (N1 + N2) a partir do qual o motivo é "Complexa"
VOLUME_MINIMO_ATENDENTE = 30     # chamados no mês para a taxa do atendente ser considerada estável
FLUXO_POR_AUTOSSERVICO = {"URA": "Telefone", "Chatbot": "Digital"}
FLUXOS_HUMANOS = ["Telefone", "Digital"]

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {gold}.exports")

log_gold = {}


def gravar_gold(df, tabela):
    # Overwrite: a Gold é reconstruída inteira a cada execução. Devolve a tabela já gravada,
    # para as etapas seguintes lerem o resultado persistido em vez de recalcular a lógica.
    destino = f"{gold}.{tabela}"
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(destino)
    resultado = spark.table(destino)
    print(f"OK: {destino}: {resultado.count()} linhas.")
    display(resultado)
    return resultado


def com_chave(df, nome_sk, ordenar_por, inicio=1):
    # Chave substituta determinística: row_number ordenado pela chave natural.
    # Para o volume do case (dezenas de milhares de linhas) uma janela única é suficiente.
    janela = Window.orderBy(*ordenar_por)
    colunas = df.columns
    return df.withColumn(nome_sk, F.row_number().over(janela) + (inicio - 1)).select(nome_sk, *colunas)


def contar(condicao):
    return F.sum(F.when(condicao, 1).otherwise(0))


def razao(numerador, denominador, casas=4):
    # Nunca divide por zero: sem denominador o indicador fica NULL, e não 0.
    return F.when(F.col(denominador) > 0, F.round(F.col(numerador) / F.col(denominador), casas))


def registrar_gold(base, regra, afetadas, total, tipo="informacao", etapa="modelagem"):
    log_gold[(base, regra)] = (int(afetadas), int(total), tipo, etapa)


def salvar_log_gold():
    # Consolida o log da Silver com as regras da Gold. A coluna "camada" diz de onde veio cada linha.
    colunas = ["camada", "base", "regra", "linhas_afetadas", "total_linhas", "percentual", "data_execucao"]
    linhas = [("Gold", base, regra, *valores) for (base, regra), valores in sorted(log_gold.items())]
    df_gold = spark.createDataFrame(
        linhas, "camada string, base string, regra string, linhas_afetadas long, total_linhas long, tipo string, etapa string"
    ).withColumn(
        "percentual", F.when(F.col("total_linhas") > 0, F.round(100.0 * F.col("linhas_afetadas") / F.col("total_linhas"), 2))
    ).withColumn("data_execucao", F.current_timestamp())
    df_silver = spark.table(f"{silver}.tb_log_qualidade").withColumn("camada", F.lit("Silver"))
    gravar_gold(df_silver.select(*colunas).unionByName(df_gold.select(*colunas)), "log_qualidade")


def exigir_gold(base, regra, afetadas, total):
    # Validação bloqueante: salva o diagnóstico e interrompe o Job antes da exportação.
    registrar_gold(base, regra, afetadas, total, "bloqueio", "validacao")
    if afetadas:
        salvar_log_gold()
        raise ValueError(f"{base}: {regra}; {afetadas} linhas. Consulte {gold}.log_qualidade.")

## Contrato de integração com a Silver

- **Validação de esquema:** verifica a presença de todas as tabelas e colunas obrigatórias da Silver antes de iniciar o processamento
- Interrompe o pipeline com exceção tratada caso alguma estrutura esperada esteja ausente

In [0]:
# 1. Mapeamento das tabelas e colunas obrigatórias da camada Silver
contrato_silver = {
    "tb_chamados": "id_chamado id_cliente id_motivo id_canal id_atendente canal tipo_canal nivel_canal resolvido "
                   "hora_abertura_chamado hora_inicio_atendimento hora_finalizacao_atendimento tempo_espera tempo_atendimento "
                   "desistencia flag_cliente_corrigido flag_atendente_inexistente flag_vinculo_atendente_invalido "
                   "flag_nivel_atendente_incompativel flag_ano_abertura_corrigido",
    "tb_custos": "id_chamado custo flag_custo_negativo_corrigido",
    "tb_pesquisa_satisfacao": "id_chamado nota_atendimento satisfeito flag_pesquisa_em_desistencia",
    "tb_clientes": "id_cliente regiao idade faixa_etaria data_cadastro dia_vencimento_fatura email",
    "tb_atendentes": "id_atendente nome_atendente nivel_atendimento status_atendente data_admissao data_desligamento",
    "tb_canais": "id_canal nome_canal tipo_canal status_canal nivel_canal",
    "tb_motivos": "id_motivo nome_motivo categoria criticidade",
    "tb_log_qualidade": "base regra linhas_afetadas total_linhas percentual data_execucao",
}

# 2. Verificação de integridade da estrutura da Silver
for tabela, colunas in contrato_silver.items():
    if not spark.catalog.tableExists(f"{silver}.{tabela}"):
        raise ValueError(f"Execute a Silver completa: falta {silver}.{tabela}.")
    faltam = [c for c in colunas.split() if c not in spark.table(f"{silver}.{tabela}").columns]
    if faltam:
        raise ValueError(f"{tabela}: faltam colunas esperadas pela Gold: {', '.join(faltam)}")
print("Contrato com a Silver conferido.")

## 2. dim_calendario

- **Período:** cobre anos completos com base nas datas de abertura dos chamados, garantindo a integridade relacional
- **sk_data:** chave substituta inteira no formato `AAAAMMDD`
- **Atributos temporais:** ano, mês, nome do mês, ano-mês, trimestre, dia da semana e indicador de final de semana
- **Flags temporais:** `flag_mes_com_dados` e `flag_mes_comparavel` (identifica meses presentes em mais de um ano)

In [0]:
# Identificar intervalo de datas presentes na base de chamados
chamados = spark.table(f"{silver}.tb_chamados")
datas = chamados.select(F.to_date("hora_abertura_chamado").alias("data")).where(F.col("data").isNotNull())
limites = datas.agg(F.min("data").alias("ini"), F.max("data").alias("fim")).first()
if limites["ini"] is None:
    raise ValueError("tb_chamados não tem datas de abertura válidas.")

# Anos completos: de 01/01 do primeiro ano a 31/12 do último ano com dados.
inicio_calendario = date(limites["ini"].year, 1, 1)
fim_calendario = date(limites["fim"].year, 12, 31)

meses_pt = ["Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
            "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"]
dias_pt = ["Segunda-feira", "Terça-feira", "Quarta-feira", "Quinta-feira", "Sexta-feira", "Sábado", "Domingo"]

# Meses que realmente têm chamados, e meses do ano presentes em 2 anos ou mais (comparáveis).
meses_com_dados = datas.select(F.year("data").alias("ano"), F.month("data").alias("mes")).distinct() \
    .withColumn("_com_dados", F.lit(True))
meses_comparaveis = meses_com_dados.groupBy("mes").agg(F.countDistinct("ano").alias("anos")) \
    .where(F.col("anos") >= 2).select("mes", F.lit(True).alias("_comparavel"))

dia_iso = ((F.dayofweek("data") + 5) % 7) + 1   # 1 = segunda ... 7 = domingo

# Construir dimensão calendário completa
dim_calendario = (
    spark.range(1).select(F.explode(F.sequence(F.lit(inicio_calendario), F.lit(fim_calendario), F.expr("INTERVAL 1 DAY"))).alias("data"))
    .withColumn("ano", F.year("data")).withColumn("mes", F.month("data"))
    .join(meses_com_dados, ["ano", "mes"], "left")
    .join(meses_comparaveis, "mes", "left")
    .select(
        F.date_format("data", "yyyyMMdd").cast("int").alias("sk_data"),
        "data", "ano", "mes",
        F.element_at(F.array(*[F.lit(m) for m in meses_pt]), F.col("mes")).alias("nome_mes"),
        F.date_format("data", "yyyy-MM").alias("ano_mes"),
        F.trunc("data", "month").alias("mes_inicio"),
        F.date_format(F.trunc("data", "month"), "yyyyMMdd").cast("int").alias("sk_data_mes"),
        F.when(F.col("data") >= F.lit(limites["ini"].replace(day=1)),
               (F.col("ano") - limites["ini"].year) * 12 + F.col("mes") - limites["ini"].month + 1).alias("indice_mes"),
        F.quarter("data").alias("trimestre"),
        F.concat(F.col("ano"), F.lit("-T"), F.quarter("data")).alias("ano_trimestre"),
        F.dayofmonth("data").alias("dia_do_mes"),
        dia_iso.alias("dia_semana_num"),
        F.element_at(F.array(*[F.lit(d) for d in dias_pt]), dia_iso).alias("dia_semana"),
        (dia_iso >= 6).alias("fim_de_semana"),
        F.coalesce("_com_dados", F.lit(False)).alias("flag_mes_com_dados"),
        (F.coalesce("_com_dados", F.lit(False)) & F.coalesce("_comparavel", F.lit(False))).alias("flag_mes_comparavel"),
    )
)
dim_calendario = gravar_gold(dim_calendario, "dim_calendario")

## 3. dim_cliente

- **Privacidade (LGPD):** remoção de `nome` e `email`, mantendo `email_valido` para auditoria de qualidade
- **sk_cliente:** chave substituta gerada sequencialmente a partir do `id_cliente`
- **Atributos:** região, idade, faixa etária, data de cadastro, ano-mês de cadastro e dia de vencimento da fatura
- **tem_chamado:** flag indicando se o cliente possui ao menos um chamado registrado no histórico

In [0]:
clientes_com_chamado = chamados.select("id_cliente").distinct().withColumn("_tem_chamado", F.lit(True))

# Montar a dimensão de clientes com substituição de chave primária e tratamento de privacidade
dim_cliente = com_chave(
    spark.table(f"{silver}.tb_clientes")
    .join(clientes_com_chamado, "id_cliente", "left")
    .select(
        "id_cliente", "regiao", "idade", "faixa_etaria", "data_cadastro",
        F.date_format("data_cadastro", "yyyy-MM").alias("ano_mes_cadastro"),
        "dia_vencimento_fatura",
        F.col("idade").isNotNull().alias("idade_valida"),
        F.col("email").isNotNull().alias("email_valido"),
        F.coalesce("_tem_chamado", F.lit(False)).alias("tem_chamado"),
    ),
    "sk_cliente", ["id_cliente"],
)
dim_cliente = gravar_gold(dim_cliente, "dim_cliente")

## 4. dim_motivo

- **sk_motivo:** chave substituta gerada a partir do `id_motivo`
- **nome_curto:** versão simplificada do nome do motivo, removendo descrições longas entre parênteses
- **Atributos:** categoria, criticidade e ordem numérica de criticidade (1: Baixa, 2: Média, 3: Alta)

In [0]:
dim_motivo = com_chave(
    spark.table(f"{silver}.tb_motivos").select(
        "id_motivo", "nome_motivo",
        F.trim(F.regexp_replace("nome_motivo", r"\s*\(.*\)\s*$", "")).alias("nome_curto"),
        "categoria", "criticidade",
        F.when(F.col("criticidade") == "Alta", 3).when(F.col("criticidade") == "Média", 2)
         .when(F.col("criticidade") == "Baixa", 1).alias("ordem_criticidade"),
    ),
    "sk_motivo", ["id_motivo"],
)
dim_motivo = gravar_gold(dim_motivo, "dim_motivo")

## 5. dim_atendente

- **Membros reais:** carregados da `tb_atendentes` com `sk_atendente` sequencial (> 0)
- **Membro de exceção `-1`:** "Não se aplica (autosserviço)" para chamados via URA ou Chatbot
- **Membro de exceção `-2`:** "Desconhecido (ID inválido)" para chamados humanos com atendente não cadastrado
- **Atributos:** nome, nível de atendimento (N1/N2), status, datas de admissão/desligamento e ano de admissão

In [0]:
# Carregar atendentes reais e gerar chaves substitutas positivas
atendentes_reais = com_chave(
    spark.table(f"{silver}.tb_atendentes").select(
        "id_atendente", "nome_atendente", "nivel_atendimento",
        F.concat(F.lit("N"), F.col("nivel_atendimento").cast("string")).alias("nivel"),
        "status_atendente", "data_admissao", "data_desligamento",
        F.year("data_admissao").alias("ano_admissao"),
    ),
    "sk_atendente", ["id_atendente"],
)

# Definir membros de exceção para autosserviço (-1) e ID inválido (-2)
membros_excecao = spark.createDataFrame(
    [(-1, None, "Não se aplica (autosserviço)", None, "Não se aplica", "Não se aplica", None, None, None),
     (-2, None, "Desconhecido (ID inválido)", None, "Desconhecido", "Desconhecido", None, None, None)],
    "sk_atendente int, id_atendente int, nome_atendente string, nivel_atendimento int, nivel string, "
    "status_atendente string, data_admissao date, data_desligamento date, ano_admissao int",
)

# Unificar atendentes reais com membros de exceção
dim_atendente = gravar_gold(atendentes_reais.unionByName(membros_excecao), "dim_atendente")

## 6. Jornada

Todo chamado recebe, por funções de janela, o `id_jornada` (o `id_chamado` do primeiro chamado da jornada, que é estável entre execuções), a posição na jornada e o fluxo.

A ordenação dentro de cliente e motivo é: hora de abertura, depois etapa do funil (autosserviço, N1, N2), depois `id_chamado`. A etapa entra como desempate para que, em chamados no mesmo segundo, o autosserviço venha antes do humano.

**Quando uma jornada começa:**
1. O chamado é URA ou Chatbot (regra do case), **ou**
2. é o primeiro do cliente naquele motivo (jornada "órfã", que começa direto no humano), **ou**
3. passou mais de `JANELA_MAX_HORAS` desde o fim do chamado anterior (salvaguarda, registrada em `flag_inicio_por_janela`).

`escalonado` é verdadeiro quando existe um próximo chamado na mesma jornada. Isso é importante para ler a resolução: um chamado intermediário aparece como "Não resolvido" **porque escalou**, e não porque o cliente ficou sem solução. A resolução que vale para o funil é a do **último chamado da jornada**.

In [0]:
# Mapear equivalência de nível do canal e dicionário de fluxos para autosserviço
etapa_funil = (F.when(F.col("nivel_canal") == "Autosserviço", 1)
                .when(F.col("nivel_canal") == "N1", 2)
                .when(F.col("nivel_canal") == "N2", 3))
fluxo_por_canal = F.create_map(*[F.lit(x) for par in FLUXO_POR_AUTOSSERVICO.items() for x in par])

ordem = ["hora_abertura_chamado", "etapa_funil", "id_chamado"]
por_cliente_motivo = Window.partitionBy("id_cliente", "id_motivo").orderBy(*ordem)

# Detecta onde cada jornada começa
sequencia = (
    chamados.withColumn("etapa_funil", etapa_funil)
    .withColumn("_fim_anterior", F.lag(F.coalesce("hora_finalizacao_atendimento", "hora_abertura_chamado")).over(por_cliente_motivo))
    .withColumn("_horas_desde_anterior",
                (F.col("hora_abertura_chamado").cast("long") - F.col("_fim_anterior").cast("long")) / 3600)
    .withColumn("flag_inicio_por_janela",
                (F.col("etapa_funil") > 1) & (F.coalesce(F.col("_horas_desde_anterior"), F.lit(0.0)) > JANELA_MAX_HORAS))
    .withColumn("_inicia", (F.col("etapa_funil") == 1) | F.col("_fim_anterior").isNull() | F.col("flag_inicio_por_janela"))
    # Soma acumulada dos inícios = número da jornada dentro do par cliente + motivo
    .withColumn("_nro_jornada", F.sum(F.when(F.col("_inicia"), 1).otherwise(0))
                .over(por_cliente_motivo.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
)

# Atribui id, posição, tamanho e fluxo a cada chamado
por_jornada = Window.partitionBy("id_cliente", "id_motivo", "_nro_jornada").orderBy(*ordem)
jornada = (
    sequencia
    .withColumn("id_jornada", F.first("id_chamado").over(por_jornada))
    .withColumn("ordem_na_jornada", F.row_number().over(por_jornada))
    .withColumn("qtd_etapas_jornada", F.count(F.lit(1)).over(Window.partitionBy("id_cliente", "id_motivo", "_nro_jornada")))
    .withColumn("escalonado", F.col("ordem_na_jornada") < F.col("qtd_etapas_jornada"))
    # Fluxo herdado do canal de entrada; jornada que começa no humano fica "Atendimento Humano"
    .withColumn("fluxo", F.coalesce(fluxo_por_canal[F.first("canal").over(por_jornada)], F.lit("Atendimento Humano")))
)

## 7. dim_canal

- **Granularidade:** combinação canal × fluxo (`Telefone` ou `Digital`)
- **sk_canal:** chave substituta gerada a partir do nível do funil, ID do canal e fluxo
- **Atributos:** nome do canal, tipo (Autosserviço/Humano), nível do canal, ordem no funil, fluxo e chave combinada (`canal_fluxo`)

In [0]:
# Mapear canais de autosserviço com fluxos fixos e canais humanos multiplicados pelos fluxos
canais = spark.table(f"{silver}.tb_canais")
colunas_canal = ["id_canal", "nome_canal", "tipo_canal", "nivel_canal", "status_canal"]

autosservico = canais.where(F.col("nivel_canal") == "Autosserviço").select(*colunas_canal) \
    .withColumn("fluxo", fluxo_por_canal[F.col("nome_canal")])
if autosservico.where(F.col("fluxo").isNull()).count():
    raise ValueError("Há canal de autosserviço sem fluxo definido em FLUXO_POR_AUTOSSERVICO.")

humano = canais.where(F.col("nivel_canal") != "Autosserviço").select(*colunas_canal) \
    .crossJoin(spark.createDataFrame([(f,) for f in FLUXOS_HUMANOS], "fluxo string"))
canal_fluxo = autosservico.unionByName(humano)

# Combinações que existem nos dados mas não estão no desenho padrão (ex.: "Atendimento Humano")
extras = (jornada.select("id_canal", "fluxo").distinct()
          .join(canal_fluxo.select("id_canal", "fluxo"), ["id_canal", "fluxo"], "left_anti")
          .join(canais.select(*colunas_canal), "id_canal", "left"))

# Montar a dimensão de canais com chaves substitutas
dim_canal = com_chave(
    canal_fluxo.unionByName(extras)
    .withColumn("ordem_funil", etapa_funil)
    .withColumn("canal_fluxo", F.concat_ws(" - ", "nome_canal", "fluxo"))
    .select("id_canal", "nome_canal", "tipo_canal", "nivel_canal", "ordem_funil", "fluxo", "canal_fluxo", "status_canal"),
    "sk_canal", ["ordem_funil", "id_canal", "fluxo"],
)
dim_canal = gravar_gold(dim_canal, "dim_canal")

## 8. `fact_chamados`

Um registro por chamado. Custo e pesquisa entram por **LEFT JOIN** (a Silver garante no máximo uma linha de cada por chamado, mas nem todo chamado tem custo ou nota). Nada é transformado em zero.

Pontos importantes:

- **Chaves.** Cada dimensão vira uma `sk_*`. O atendente usa `-1` (autosserviço) e `-2` (ID inválido).
- **`tempo_casa_meses`** é o tempo de casa do atendente **na data do chamado**. Fica NULL quando o atendimento é anterior à admissão ou posterior ao desligamento, porque o valor negativo ou inflado distorceria a análise de capacitação.
- **Flags de qualidade** (`flag_*`) carregam para a Gold os problemas tratados na Silver. Assim é possível filtrar no Power BI e mostrar o impacto real de cada inconsistência.
- `flag_custo_ausente` cobre custo não registrado, zero e inválido; `flag_custo_corrigido` marca o erro de sinal corrigido.

In [0]:
# Carregar tabelas complementares de custos, pesquisas e atendentes
custos = spark.table(f"{silver}.tb_custos").select("id_chamado", "custo", "flag_custo_negativo_corrigido")
pesquisas = spark.table(f"{silver}.tb_pesquisa_satisfacao") \
    .select("id_chamado", "nota_atendimento", "satisfeito", "flag_pesquisa_em_desistencia")

atendentes_chave = dim_atendente.where(F.col("sk_atendente") > 0) \
    .select("id_atendente", F.col("sk_atendente").alias("_sk_atendente"), "data_admissao")

# Construir a tabela fato de chamados relacionando dimensões e métricas
fact_chamados = (
    jornada
    .join(custos, "id_chamado", "left")
    .join(pesquisas, "id_chamado", "left")
    .join(dim_cliente.select("id_cliente", "sk_cliente"), "id_cliente", "left")
    .join(dim_motivo.select("id_motivo", "sk_motivo"), "id_motivo", "left")
    .join(dim_canal.select("id_canal", "fluxo", "sk_canal"), ["id_canal", "fluxo"], "left")
    .join(atendentes_chave, "id_atendente", "left")
    .withColumn("sk_atendente",
                F.when(F.col("tipo_canal") == "Autosserviço", F.lit(-1))
                 .when(F.col("_sk_atendente").isNull(), F.lit(-2))
                 .otherwise(F.col("_sk_atendente")))
    .withColumn("tempo_casa_meses",
                F.when(~F.coalesce(F.col("flag_vinculo_atendente_invalido"), F.lit(False)) & F.col("data_admissao").isNotNull(),
                       F.floor(F.months_between(F.to_date("hora_abertura_chamado"), F.col("data_admissao")))).cast("int"))
    .select(
        "id_chamado", "id_jornada", "ordem_na_jornada", "qtd_etapas_jornada",
        "sk_cliente", "sk_motivo", "sk_canal", "sk_atendente",
        F.date_format("hora_abertura_chamado", "yyyyMMdd").cast("int").alias("sk_data"),
        F.col("hora_abertura_chamado").alias("hora_abertura"),
        F.col("hora_inicio_atendimento").alias("hora_inicio"),
        F.col("hora_finalizacao_atendimento").alias("hora_finalizacao"),
        F.hour("hora_abertura_chamado").alias("hora_do_dia"),
        F.col("tempo_espera").alias("tempo_espera_seg"),
        F.col("tempo_atendimento").alias("tempo_atendimento_seg"),
        "custo", "nota_atendimento", "satisfeito", "resolvido", "desistencia", "escalonado", "tempo_casa_meses",
        F.col("custo").isNull().alias("flag_custo_ausente"),
        F.coalesce("flag_custo_negativo_corrigido", F.lit(False)).alias("flag_custo_corrigido"),
        F.coalesce("flag_pesquisa_em_desistencia", F.lit(False)).alias("flag_nota_em_desistencia"),
        F.coalesce("flag_vinculo_atendente_invalido", F.lit(False)).alias("flag_vinculo_atendente_invalido"),
        "flag_atendente_inexistente", "flag_nivel_atendente_incompativel", "flag_cliente_corrigido",
        "flag_ano_abertura_corrigido", "flag_inicio_por_janela",
    )
)
fact_chamados = gravar_gold(fact_chamados, "fact_chamados")

## 9. `fact_jornadas`

Uma linha por jornada, agregada a partir do `fact_chamados` já gravado. Cuidados que mudam os números:

- **Resolução final** é a do **último** chamado (`max_by` pela posição), em três estados: Sim, Não e Não informado. "Não informado" não é tratado como "Não".
- **Nota final** é a do último chamado **com nota válida**. Nem todo chamado tem pesquisa.
- **Custo total** soma só os custos registrados. `qtd_chamados_sem_custo` e `custo_completo` dizem se o total está subestimado.
- **`nivel_maximo`** é a etapa mais alta atingida (Autosserviço, N1 ou N2) e é a base do funil e do custo por nível.
- **`resolvida_no_autosservico`** exige resolução final "Sim" em jornada que não passou do autosserviço.

In [0]:
# Carregar mapeamento de níveis do canal
niveis_canal = dim_canal.select("sk_canal", "nivel_canal", "ordem_funil")

# Agregar chamados no nível de jornada
fact_jornadas = (
    fact_chamados.join(niveis_canal, "sk_canal", "left")
    .groupBy("id_jornada")
    .agg(
        F.min("sk_cliente").alias("sk_cliente"),
        F.min("sk_motivo").alias("sk_motivo"),
        F.min_by("sk_canal", "ordem_na_jornada").alias("sk_canal_entrada"),   # canal e fluxo de entrada
        F.min("sk_data").alias("sk_data_inicio"),
        F.min("hora_abertura").alias("hora_inicio_jornada"),
        F.max("hora_finalizacao").alias("hora_fim_jornada"),
        F.count(F.lit(1)).alias("qtd_etapas"),
        F.max_by("nivel_canal", "ordem_funil").alias("nivel_maximo"),
        F.max(F.col("nivel_canal") == "N1").alias("passou_n1"),
        F.max(F.col("nivel_canal") == "N2").alias("passou_n2"),
        (F.min_by("nivel_canal", "ordem_na_jornada") != "Autosserviço").alias("jornada_sem_autosservico"),
        F.sum(F.col("desistencia").cast("int")).alias("qtd_desistencias"),
        F.sum("tempo_espera_seg").alias("espera_total_seg"),
        F.sum("tempo_atendimento_seg").alias("atendimento_total_seg"),
        F.sum("custo").cast("decimal(18,2)").alias("custo_total"),
        F.sum(F.col("flag_custo_ausente").cast("int")).alias("qtd_chamados_sem_custo"),
        F.max_by("resolvido", "ordem_na_jornada").alias("resolucao_final"),
        F.max(F.when(F.col("nota_atendimento").isNotNull(),
                     F.struct("ordem_na_jornada", "nota_atendimento")))["nota_atendimento"].alias("nota_final"),
    )
    .withColumn("duracao_total_seg", F.col("hora_fim_jornada").cast("long") - F.col("hora_inicio_jornada").cast("long"))
    .withColumn("custo_completo", F.col("qtd_chamados_sem_custo") == 0)
    .withColumn("resolvida_no_autosservico", (F.col("nivel_maximo") == "Autosserviço") & (F.col("resolucao_final") == "Sim"))
    .withColumn("satisfeita_final", F.when(F.col("nota_final").isNotNull(), F.col("nota_final") >= 4))
    .select(
        "id_jornada", "sk_cliente", "sk_motivo", "sk_canal_entrada", "sk_data_inicio",
        "hora_inicio_jornada", "hora_fim_jornada", "duracao_total_seg",
        "qtd_etapas", "nivel_maximo", "passou_n1", "passou_n2", "jornada_sem_autosservico", "qtd_desistencias",
        "espera_total_seg", "atendimento_total_seg", "custo_total", "qtd_chamados_sem_custo", "custo_completo",
        "resolucao_final", "resolvida_no_autosservico", "nota_final", "satisfeita_final",
    )
)
fact_jornadas = gravar_gold(fact_jornadas, "fact_jornadas")

## 10. Indicadores

Os KPIs gravam **numeradores e denominadores** junto das taxas. No Power BI, some os numeradores e divida pelos denominadores; nunca faça média das taxas. Denominadores adotados:

| Indicador | Fórmula | Denominador |
|---|---|---|
| CSAT | satisfeitos ÷ com nota | só notas válidas (1 a 5) |
| Taxa de resolução | resolvidos ÷ resolução informada | exclui "Não informado" |
| TME e TMA | soma do tempo ÷ chamados com tempo | exclui desistências (sem tempo). Para comparar só o atendimento humano, filtre o nível no `dim_canal` |
| Desistência | desistências ÷ chamados | todos os chamados |
| Escalonamento | escalonados ÷ chamados | todos os chamados do grupo |
| Custo médio | custo total ÷ chamados com custo | exclui custo ausente |

Todas as taxas são frações (0 a 1) e os tempos estão em segundos.

In [0]:
# Função geradora das agregações básicas de volume, tempos, notas e custos
def agregados():
    return [
        F.count(F.lit(1)).alias("qtd_chamados"),
        contar(F.col("desistencia")).alias("qtd_desistencias"),
        contar(~F.col("desistencia")).alias("qtd_atendidos"),
        contar(F.col("escalonado")).alias("qtd_escalonados"),
        contar(F.col("resolvido") == "Sim").alias("qtd_resolvidos"),
        contar(F.col("resolvido").isin("Sim", "Não")).alias("qtd_resolucao_informada"),
        contar(F.col("nota_atendimento").isNotNull()).alias("qtd_com_nota"),
        contar(F.col("satisfeito")).alias("qtd_satisfeitos"),
        F.sum("nota_atendimento").alias("soma_notas"),
        contar(F.col("tempo_espera_seg").isNotNull()).alias("qtd_com_espera"),
        F.sum("tempo_espera_seg").alias("soma_espera_seg"),
        contar(F.col("tempo_atendimento_seg").isNotNull()).alias("qtd_com_atendimento"),
        F.sum("tempo_atendimento_seg").alias("soma_atendimento_seg"),
        contar(F.col("custo").isNotNull()).alias("qtd_com_custo"),
        F.sum("custo").cast("decimal(18,2)").alias("custo_total"),
    ]

# Função de cálculo de taxas, médias e percentuais a partir dos totais agregados
def com_taxas(df):
    return (
        df.withColumn("pct_desistencia", razao("qtd_desistencias", "qtd_chamados"))
        .withColumn("pct_escalonamento", razao("qtd_escalonados", "qtd_chamados"))
        .withColumn("taxa_resolucao", razao("qtd_resolvidos", "qtd_resolucao_informada"))
        .withColumn("csat", razao("qtd_satisfeitos", "qtd_com_nota"))
        .withColumn("nota_media", razao("soma_notas", "qtd_com_nota", 2))
        .withColumn("tme_seg", razao("soma_espera_seg", "qtd_com_espera", 1))
        .withColumn("tma_seg", razao("soma_atendimento_seg", "qtd_com_atendimento", 1))
        .withColumn("custo_medio", razao("custo_total", "qtd_com_custo", 2))
        .withColumn("cobertura_custo", razao("qtd_com_custo", "qtd_chamados"))
        .withColumn("cobertura_nota", razao("qtd_com_nota", "qtd_chamados"))
    )


mes_abertura = F.trunc(F.to_date("hora_abertura"), "month")

**`kpi_mensal`**: mês × canal × fluxo. `sk_data_mes` aponta para o primeiro dia do mês em `dim_calendario`, então o painel executivo se relaciona ao calendário sem tabela extra. Para comparar custo com o crescimento da base (não há receita nos dados), use no Power BI a contagem acumulada de clientes por `data_cadastro` em `dim_cliente`.

In [0]:
# Gerar agregações mensais por canal e fluxo
kpi_mensal = com_taxas(
    fact_chamados.groupBy(mes_abertura.alias("mes"), "sk_canal").agg(*agregados())
).select(
    F.date_format("mes", "yyyyMMdd").cast("int").alias("sk_data_mes"), "mes", F.date_format("mes", "yyyy-MM").alias("ano_mes"),
    "sk_canal", "qtd_chamados", "qtd_atendidos", "qtd_desistencias", "qtd_escalonados", "qtd_resolvidos",
    "qtd_resolucao_informada", "qtd_com_nota", "qtd_satisfeitos", "soma_notas", "qtd_com_espera", "soma_espera_seg",
    "qtd_com_atendimento", "soma_atendimento_seg", "qtd_com_custo", "custo_total",
    "pct_desistencia", "pct_escalonamento", "taxa_resolucao", "csat", "nota_media", "tme_seg", "tma_seg",
    "custo_medio", "cobertura_custo", "cobertura_nota",
)
kpi_mensal = gravar_gold(kpi_mensal, "kpi_mensal")

**`kpi_funil_motivo`** responde às duas perguntas centrais do funil:

- **Demandas simples que chegam ao humano sem necessidade:** `perfil_demanda = "Simples"` com `pct_chegou_humano` alto. São candidatas a ampliar o autosserviço.
- **Demandas complexas que poderiam ir direto ao N2:** `perfil_demanda = "Complexa"`. A maioria das jornadas humanas desse motivo acaba no N2.

O perfil usa `taxa_n1_para_n2 = fim no N2 ÷ (fim no N1 + fim no N2)` contra `LIMITE_COMPLEXA`. Os custos médios por nível final usam só jornadas com custo completo, para não subestimar. A economia de resolver mais cedo é a diferença entre esses custos vezes o volume, calculada no painel.

In [0]:
# Definir condições de nível máximo e integridade de custo
def media_se(condicao, coluna):
    return F.avg(F.when(condicao, F.col(coluna)))

fim_auto = F.col("nivel_maximo") == "Autosserviço"
fim_n1 = F.col("nivel_maximo") == "N1"
fim_n2 = F.col("nivel_maximo") == "N2"
custo_ok = F.col("custo_completo")

# Construir matriz do funil por motivo e classificar perfil de demanda
kpi_funil_motivo = (
    fact_jornadas.groupBy("sk_motivo").agg(
        F.count(F.lit(1)).alias("qtd_jornadas"),
        contar(fim_auto).alias("qtd_fim_autosservico"),
        contar(fim_n1).alias("qtd_fim_n1"),
        contar(fim_n2).alias("qtd_fim_n2"),
        contar(F.col("resolvida_no_autosservico")).alias("qtd_resolvidas_autosservico"),
        contar(F.col("resolucao_final") == "Sim").alias("qtd_resolvidas_final"),
        F.sum("custo_total").cast("decimal(18,2)").alias("custo_total"),
        F.round(media_se(fim_auto & custo_ok, "custo_total"), 2).alias("custo_medio_fim_autosservico"),
        F.round(media_se(fim_n1 & custo_ok, "custo_total"), 2).alias("custo_medio_fim_n1"),
        F.round(media_se(fim_n2 & custo_ok, "custo_total"), 2).alias("custo_medio_fim_n2"),
        F.round(media_se(fim_auto, "nota_final"), 2).alias("nota_media_fim_autosservico"),
        F.round(media_se(fim_n1, "nota_final"), 2).alias("nota_media_fim_n1"),
        F.round(media_se(fim_n2, "nota_final"), 2).alias("nota_media_fim_n2"),
    )
    .withColumn("qtd_chegou_humano", F.col("qtd_fim_n1") + F.col("qtd_fim_n2"))
    .withColumn("pct_resolvida_autosservico", razao("qtd_resolvidas_autosservico", "qtd_jornadas"))
    .withColumn("pct_chegou_humano", razao("qtd_chegou_humano", "qtd_jornadas"))
    .withColumn("pct_chegou_n2", razao("qtd_fim_n2", "qtd_jornadas"))
    .withColumn("taxa_n1_para_n2", razao("qtd_fim_n2", "qtd_chegou_humano"))
    .withColumn("taxa_resolucao_final", razao("qtd_resolvidas_final", "qtd_jornadas"))
    .withColumn("perfil_demanda",
                F.when(F.col("taxa_n1_para_n2").isNull(), F.lit("Sem passagem pelo humano"))
                 .when(F.col("taxa_n1_para_n2") >= LIMITE_COMPLEXA, F.lit("Complexa"))
                 .otherwise(F.lit("Simples")))
)
kpi_funil_motivo = gravar_gold(kpi_funil_motivo, "kpi_funil_motivo")

**`kpi_atendente_mensal`**: só atendentes reais (`sk_atendente > 0`); os membros `-1` e `-2` ficam de fora do ranking. `volume_minimo_ok` indica se o atendente teve chamados suficientes no mês (`VOLUME_MINIMO_ATENDENTE`) para a taxa ser considerada estável. Filtre por ele ao procurar outliers, para não destacar quem só teve poucos chamados.

In [0]:
# Gerar indicadores mensais exclusivamente para atendentes reais
kpi_atendente_mensal = com_taxas(
    fact_chamados.where(F.col("sk_atendente") > 0)
    .groupBy(mes_abertura.alias("mes"), "sk_atendente").agg(*agregados())
).withColumn("volume_minimo_ok", F.col("qtd_chamados") >= VOLUME_MINIMO_ATENDENTE).select(
    F.date_format("mes", "yyyyMMdd").cast("int").alias("sk_data_mes"), "mes", F.date_format("mes", "yyyy-MM").alias("ano_mes"),
    "sk_atendente", "qtd_chamados", "qtd_atendidos", "qtd_desistencias", "qtd_escalonados", "qtd_resolvidos",
    "qtd_resolucao_informada", "qtd_com_nota", "qtd_satisfeitos", "soma_notas", "qtd_com_espera", "soma_espera_seg",
    "qtd_com_atendimento", "soma_atendimento_seg",
    "pct_desistencia", "pct_escalonamento", "taxa_resolucao", "csat", "nota_media", "tme_seg", "tma_seg",
    "volume_minimo_ok",
)
kpi_atendente_mensal = gravar_gold(kpi_atendente_mensal, "kpi_atendente_mensal")

## 11. Validações e log de qualidade

Duas famílias de regras:

- **Bloqueantes:** se qualquer uma falhar, o log é salvo e o Job para **antes da exportação**. O Power BI continua com os últimos CSVs válidos, em vez de receber números errados.
- **Informativas e alertas:** não param o Job, mas entram no `log_qualidade` para sustentar a leitura de confiabilidade na apresentação. Ao contrário dos percentuais da Silver, que podem atingir as mesmas linhas, aqui cada regra tem o seu próprio denominador.

A reconciliação é feita **linha a linha** (custo e nota do fato contra a Silver), não só por totais. Um total pode bater por compensação de erros.

In [0]:
# Carregar totais da Silver para validação cruzada
silver_chamados = spark.table(f"{silver}.tb_chamados")
total_chamados = silver_chamados.count()
total_fato = fact_chamados.count()

# ---- Bloqueantes: integridade do modelo

# Executar validações bloqueantes de contagem e unicidade de chaves
exigir_gold("fact_chamados", "Quantidade de chamados diferente da Silver", abs(total_fato - total_chamados), total_chamados)
exigir_gold("fact_chamados", "id_chamado repetido",
            total_fato - fact_chamados.select("id_chamado").distinct().count(), total_fato)

# Validar integridade referencial das dimensões com a fato de chamados
for nome_dim, dim, chave in [("dim_cliente", dim_cliente, "sk_cliente"), ("dim_motivo", dim_motivo, "sk_motivo"),
                             ("dim_canal", dim_canal, "sk_canal"), ("dim_atendente", dim_atendente, "sk_atendente")]:
    exigir_gold(nome_dim, "Chave substituta repetida", dim.count() - dim.select(chave).distinct().count(), dim.count())
    orfas = fact_chamados.join(dim.select(chave), chave, "left_anti").count()
    exigir_gold("fact_chamados", f"Chamados sem linha em {nome_dim} (FK orfa)", orfas, total_fato)

orfas_data = fact_chamados.join(dim_calendario.select(F.col("sk_data")), "sk_data", "left_anti").count()
exigir_gold("fact_chamados", "Chamados sem linha em dim_calendario (FK orfa)", orfas_data, total_fato)

# Validar integridade referencial da fato de jornadas
total_jornadas = fact_jornadas.count()
exigir_gold("fact_jornadas", "id_jornada repetido",
            total_jornadas - fact_jornadas.select("id_jornada").distinct().count(), total_jornadas)
exigir_gold("fact_jornadas", "Soma das etapas diferente do total de chamados",
            abs((fact_jornadas.agg(F.sum("qtd_etapas")).first()[0] or 0) - total_fato), total_jornadas)
for nome_dim, dim, chave_fato in [("dim_cliente", dim_cliente, "sk_cliente"), ("dim_motivo", dim_motivo, "sk_motivo"),
                                  ("dim_canal", dim_canal, "sk_canal_entrada")]:
    chave_dim = [c for c in dim.columns if c.startswith("sk_")][0]
    orfas = fact_jornadas.join(dim.select(F.col(chave_dim).alias(chave_fato)), chave_fato, "left_anti").count()
    exigir_gold("fact_jornadas", f"Jornadas sem linha em {nome_dim} (FK orfa)", orfas, total_jornadas)

# ---- Bloqueantes: reconciliação linha a linha com a Silver
custo_silver = spark.table(f"{silver}.tb_custos").select("id_chamado", F.col("custo").alias("_custo_silver"))
div_custo = fact_chamados.select("id_chamado", "custo").join(custo_silver, "id_chamado", "left") \
    .where(~F.col("custo").eqNullSafe(F.col("_custo_silver"))).count()
exigir_gold("fact_chamados", "Custo diferente da Silver", div_custo, total_fato)

nota_silver = spark.table(f"{silver}.tb_pesquisa_satisfacao").select("id_chamado", F.col("nota_atendimento").alias("_nota_silver"))
div_nota = fact_chamados.select("id_chamado", "nota_atendimento").join(nota_silver, "id_chamado", "left") \
    .where(~F.col("nota_atendimento").eqNullSafe(F.col("_nota_silver"))).count()
exigir_gold("fact_chamados", "Nota diferente da Silver", div_nota, total_fato)

soma_fato = fact_chamados.agg(F.sum("custo")).first()[0]
soma_kpi = kpi_mensal.agg(F.sum("custo_total")).first()[0]
exigir_gold("kpi_mensal", "Custo total diferente do fato", int(soma_fato != soma_kpi), 1)
exigir_gold("kpi_mensal", "Quantidade de chamados diferente do fato",
            abs((kpi_mensal.agg(F.sum("qtd_chamados")).first()[0] or 0) - total_fato), total_fato)
exigir_gold("kpi_funil_motivo", "Jornadas diferentes do fact_jornadas",
            abs((kpi_funil_motivo.agg(F.sum("qtd_jornadas")).first()[0] or 0) - total_jornadas), total_jornadas)

# ---- Informativas e alertas (não bloqueiam)
# Registrar regras informativas no log de qualidade
registrar_gold("fact_chamados", "Chamados sem custo valido (ausente, zero ou invalido)",
               fact_chamados.where("flag_custo_ausente").count(), total_fato, "alerta")
registrar_gold("fact_chamados", "Chamados sem nota valida de satisfacao",
               fact_chamados.where(F.col("nota_atendimento").isNull()).count(), total_fato, "informacao")
registrar_gold("fact_chamados", "Chamados humanos com atendente desconhecido (ID invalido)",
               fact_chamados.where(F.col("sk_atendente") == -2).count(), total_fato, "alerta")
registrar_gold("fact_chamados", "Chamados que iniciaram jornada pela janela maxima",
               fact_chamados.where("flag_inicio_por_janela").count(), total_fato, "informacao")
registrar_gold("fact_jornadas", "Jornadas iniciadas direto no humano (sem autosservico antes)",
               fact_jornadas.where("jornada_sem_autosservico").count(), total_jornadas, "alerta")
registrar_gold("fact_jornadas", "Jornadas com custo incompleto (total subestimado)",
               fact_jornadas.where(~F.col("custo_completo")).count(), total_jornadas, "alerta")
registrar_gold("fact_jornadas", "Jornadas sem resolucao final informada",
               fact_jornadas.where(F.col("resolucao_final") == "Não informado").count(), total_jornadas, "alerta")

# Salvar e exibir log consolidado
salvar_log_gold()
display(spark.table(f"{gold}.log_qualidade").where(F.col("camada") == "Gold").orderBy("base", "regra"))

## 12. Exportação dos CSVs para o Power BI

Chega aqui só quem passou em todas as validações. Cada tabela vira **um único CSV** no Volume `exports`, com cabeçalho, UTF-8, ponto decimal, datas em ISO (`aaaa-MM-dd`) e aspas escapadas no padrão RFC 4180, para evitar problemas de configuração regional no Power BI.

O Spark grava uma pasta com o arquivo `part-*.csv`; copiamos esse arquivo para o nome final e apagamos a pasta temporária. Depois, cada CSV é relido e comparado com a tabela de origem.

In [0]:
# Lista das tabelas da Gold elegíveis para exportação
tabelas_exportadas = [
    "dim_cliente", "dim_canal", "dim_motivo", "dim_atendente", "dim_calendario",
    "fact_chamados", "fact_jornadas", "kpi_mensal", "kpi_funil_motivo", "kpi_atendente_mensal", "log_qualidade",
]

# Função de exportação individual com validação de contagem de linhas
def exportar_csv(nome):
    destino = f"{caminho_exports}{nome}.csv"
    temporario = f"{caminho_exports}_tmp_{nome}"
    tabela = spark.table(f"{gold}.{nome}")
    (tabela.orderBy(tabela.columns[0]).coalesce(1).write.mode("overwrite")
        .option("header", True).option("encoding", "UTF-8")
        .option("quote", '"').option("escape", '"')
        .option("dateFormat", "yyyy-MM-dd").option("timestampFormat", "yyyy-MM-dd HH:mm:ss")
        .csv(temporario))
    partes = [arquivo.path for arquivo in dbutils.fs.ls(temporario) if arquivo.name.startswith("part-") and arquivo.name.endswith(".csv")]
    if len(partes) != 1:
        raise ValueError(f"{nome}: esperado 1 arquivo CSV, encontrado {len(partes)}.")
    dbutils.fs.rm(destino)
    dbutils.fs.cp(partes[0], destino)
    dbutils.fs.rm(temporario, True)

    lidas = spark.read.option("header", True).option("encoding", "UTF-8").csv(destino).count()
    esperadas = tabela.count()
    if lidas != esperadas:
        raise ValueError(f"{nome}: CSV com {lidas} linhas, tabela com {esperadas}.")
    print(f"CSV: {destino}: {lidas} linhas.")

# Executar exportação de todas as tabelas
for nome in tabelas_exportadas:
    exportar_csv(nome)

## 13. Resumo final de execução

- Exibe a contagem consolidada de registros de todas as tabelas Delta gravadas na camada Gold
- Apresenta o resumo do funil de atendimento por nível máximo alcançado

In [0]:
# Imprimir total de registros gravados por tabela na Gold
for nome in tabelas_exportadas:
    print(f"{gold}.{nome}: {spark.table(f'{gold}.{nome}').count()} linhas.")

# Exibir visão resumida do funil por nível máximo
print("\nFunil de jornadas por nível final:")
display(
    fact_jornadas.groupBy("nivel_maximo").agg(
        F.count(F.lit(1)).alias("jornadas"),
        F.round(F.avg("nota_final"), 2).alias("nota_media"),
        F.round(F.avg(F.when(F.col("custo_completo"), F.col("custo_total"))), 2).alias("custo_medio"),
    ).orderBy("nivel_maximo")
)

## Contrato para o Power BI

| Relacionamento | Cardinalidade | Observação |
|---|---|---|
| `fact_chamados[sk_cliente, sk_motivo, sk_canal, sk_atendente, sk_data]` → dimensões | muitos para um | Padrão |
| `fact_jornadas[sk_cliente, sk_motivo, sk_canal_entrada, sk_data_inicio]` → dimensões | muitos para um | `sk_canal_entrada` aponta para `dim_canal`; `sk_data_inicio` para `dim_calendario[sk_data]` |
| `kpi_mensal[sk_canal]` e `kpi_atendente_mensal[sk_atendente]` → dimensões | muitos para um | `sk_data_mes` → `dim_calendario[sk_data]` |
| `kpi_funil_motivo[sk_motivo]` → `dim_motivo` | um para um | |
| `fact_jornadas` ↔ `fact_chamados` por `id_jornada` | **inativo** | Duas rotas ativas para as mesmas dimensões criam caminhos ambíguos; use a relação só em medidas específicas |

Uma tabela mais detalhada

| Tabela Origem | Tabela Destino | Chave de Ligação | Cardinalidade | Observação |
|---|---|---|---|---|
| `fact_chamados` | `dim_cliente` | `sk_cliente` | Muitos para Um | Padrão |
| `fact_chamados` | `dim_motivo` | `sk_motivo` | Muitos para Um | Padrão |
| `fact_chamados` | `dim_canal` | `sk_canal` | Muitos para Um | Padrão |
| `fact_chamados` | `dim_atendente` | `sk_atendente` | Muitos para Um | Padrão |
| `fact_chamados` | `dim_calendario` | `sk_data` | Muitos para Um | Padrão |
| `fact_jornadas` | `dim_cliente` | `sk_cliente` | Muitos para Um | Padrão |
| `fact_jornadas` | `dim_motivo` | `sk_motivo` | Muitos para Um | Padrão |
| `fact_jornadas` | `dim_canal` | `sk_canal_entrada` | Muitos para Um | Relacionamento via canal de entrada |
| `fact_jornadas` | `dim_calendario` | `sk_data_inicio` | Muitos para Um | Relacionamento via data de início |
| `kpi_mensal` | `dim_canal` | `sk_canal` | Muitos para Um | Relacionamento com dimensão canal |
| `kpi_mensal` | `dim_calendario` | `sk_data_mes` | Muitos para Um | Relacionamento com primeiro dia do mês |
| `kpi_atendente_mensal` | `dim_atendente` | `sk_atendente` | Muitos para Um | Relacionamento com atendentes |
| `kpi_funil_motivo` | `dim_motivo` | `sk_motivo` | Um para Um | Relacionamento direto por motivo |
| `fact_jornadas` | `fact_chamados` | `id_jornada` | Um para Muitos | **Inativo** (utilizar apenas via DAX `USERELATIONSHIP`) |

Importe `sk_*` como número inteiro, datas como data e as colunas `flag_*` como verdadeiro/falso, fixando os tipos no Power Query.